[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C16_Generative_Models_Course/02_vae/02_vae.ipynb)

# 02 · 变分自编码器 VAE（用 numpy 从零）

目标：从零实现一个 **VAE**（含 ELBO、重参数化、闭式 KL、采样、手写反向传播），在玩具 2D 分布上训练并 **从先验采样生成**；演示 **后验坍塌**。

路线：编码器输出 μ/logσ² → **重参数化 + 梯度可穿过** → **闭式 KL 对拍蒙特卡洛** → 完整 ELBO 前向+反向（梯度检验）→ 训练并采样 → 后验坍塌 → ✏️ 练习（ELBO 推导项、重参数、KL 项、隐空间采样）→ 📖 答案 → 🧪 真实超参胶囊。

> 心智模型：**编码器输出一团云 q(z|x)，KL 把云拉向先验 N(0,I)，于是从先验采样能解码出新样本**。多出的 KL 项 = 能采样的代价。

## 1 · 概率编码器：输出 μ 和 logσ²（而非一个点）

VAE 的编码器对每个 `x` 输出后验 `q(z|x)=N(μ(x), σ²(x))` 的参数。
实现上输出 **logσ²**（log-variance）而非 σ：这样 `σ=exp(0.5*logσ²)` 自动恒正，数值稳定。

玩具数据：2D 双月牙（two moons 的简化版），数据有清晰的低维结构，适合看隐空间。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def make_moons_2d(n=600, noise=0.08, seed=1):
    g = np.random.default_rng(seed)
    t = g.uniform(0, np.pi, size=n//2)
    up = np.stack([np.cos(t), np.sin(t)], 1)                       # 上半月
    dn = np.stack([1 - np.cos(t), -np.sin(t) + 0.3], 1)            # 下半月
    X = np.concatenate([up, dn], 0) + noise * g.standard_normal((n, 2))
    return (X - X.mean(0)) / X.std(0)

def encoder_forward(x, params):
    '''x:(N,2) -> 隐层 h=tanh(x@W1+b1) -> mu=h@Wmu+bmu, logvar=h@Wlv+blv'''
    h = np.tanh(x @ params['W1'] + params['b1'])
    mu = h @ params['Wmu'] + params['bmu']
    logvar = h @ params['Wlv'] + params['blv']
    return mu, logvar, h

def init_vae(D=2, H=16, d=2, seed=0):
    g = np.random.default_rng(seed); s = 0.3
    P = {}
    P['W1'] = s*g.standard_normal((D, H)); P['b1'] = np.zeros(H)
    P['Wmu'] = s*g.standard_normal((H, d)); P['bmu'] = np.zeros(d)
    P['Wlv'] = s*g.standard_normal((H, d)); P['blv'] = np.zeros(d)
    P['Wd1'] = s*g.standard_normal((d, H)); P['bd1'] = np.zeros(H)
    P['Wd2'] = s*g.standard_normal((H, D)); P['bd2'] = np.zeros(D)
    return P

X = make_moons_2d()
P = init_vae()
mu, logvar, h = encoder_forward(X, P)
print('x', X.shape, '-> mu', mu.shape, 'logvar', logvar.shape)
print('初始 σ 范围: [%.3f, %.3f]' % (np.exp(0.5*logvar).min(), np.exp(0.5*logvar).max()))
assert mu.shape == (X.shape[0], 2) and logvar.shape == mu.shape
assert np.all(np.exp(0.5*logvar) > 0), 'σ 必须恒正'
print('✅ 概率编码器：每个 x -> 一团高斯云 (μ, σ)')

## 2 · 重参数化技巧：让梯度穿过采样

直接 `z~N(μ,σ²)` 梯度断了。重参数化：`z = μ + σ⊙ε`，`ε~N(0,I)`，随机性挪到与参数无关的 `ε`。

验证两件事：① 采出的 `z` 分布与直接采样一致；② 梯度能穿过——`dz/dμ=1`、`dz/dσ=ε`，对 logvar 则 `dz/dlogvar = 0.5*σ*ε`。

In [ ]:
def reparameterize(mu, logvar, eps):
    std = np.exp(0.5 * logvar)
    return mu + std * eps                      # z = μ + σ⊙ε

# ① 分布一致性：大量样本下，重参数采样的均值/方差 ≈ (μ, σ²)
mu0 = np.array([1.0, -2.0]); logvar0 = np.log(np.array([0.25, 4.0]))   # σ=0.5, 2.0
eps = rng.standard_normal((50000, 2))
z = reparameterize(mu0, logvar0, eps)
print('重参数样本 均值≈', np.round(z.mean(0), 2), ' 目标', mu0)
print('重参数样本 方差≈', np.round(z.var(0), 2), ' 目标', np.exp(logvar0))
assert np.allclose(z.mean(0), mu0, atol=0.05)
assert np.allclose(z.var(0), np.exp(logvar0), atol=0.1)
# ② 梯度可穿过：解析 dz/dlogvar 对拍数值
e1 = np.array([0.7, -0.3])
def z_of_logvar(lv):
    return reparameterize(mu0, lv, e1)
ana = 0.5 * np.exp(0.5*logvar0) * e1                # dz/dlogvar = 0.5*σ*ε
num = (z_of_logvar(logvar0 + 1e-6) - z_of_logvar(logvar0 - 1e-6)) / (2e-6)
print('dz/dlogvar 解析', np.round(ana,4), ' 数值', np.round(num,4))
assert np.allclose(ana, num, atol=1e-6)
print('✅ 重参数化：分布一致 + 梯度可穿过 (随机性挪到 ε，μ/σ 可微)')

## 3 · 高斯 KL 闭式解 对拍 蒙特卡洛

`KL(N(μ,σ²)||N(0,1)) = 0.5*Σ(μ²+σ²-1-logσ²)`，有解析式、无需采样。

用蒙特卡洛 `E_q[log q - log p]` 估一遍，验证闭式正确；并验证 `μ=0,σ=1` 时 KL=0。

In [ ]:
def kl_closed(mu, logvar):
    '''闭式 KL(N(mu,σ²)||N(0,1))，对样本求和的每行 KL；这里给单个分布。'''
    return 0.5 * np.sum(mu**2 + np.exp(logvar) - 1.0 - logvar)

def gauss_logpdf(z, mu, logvar):
    return -0.5 * (np.log(2*np.pi) + logvar + (z - mu)**2 / np.exp(logvar))

mu1 = np.array([0.5, -1.0]); logvar1 = np.log(np.array([0.5, 2.0]))
# 蒙特卡洛：E_q[log q(z) - log p(z)]，p=N(0,1)
zc = mu1 + np.exp(0.5*logvar1) * rng.standard_normal((200000, 2))
log_q = gauss_logpdf(zc, mu1, logvar1).sum(1)
log_p = gauss_logpdf(zc, np.zeros(2), np.zeros(2)).sum(1)
kl_mc = np.mean(log_q - log_p)
kl_cf = kl_closed(mu1, logvar1)
print('KL 闭式      = %.4f' % kl_cf)
print('KL 蒙特卡洛  = %.4f' % kl_mc)
assert abs(kl_cf - kl_mc) < 0.02, '闭式应与蒙特卡洛一致'
assert abs(kl_closed(np.zeros(2), np.zeros(2))) < 1e-12, 'μ=0,σ=1 时 KL=0'
print('✅ 闭式 KL 正确（= 蒙特卡洛；后验=先验时为 0），且零方差、便宜')

## 4 · 完整 ELBO：前向 + 手写反向 + 梯度检验

解码器：`z -> tanh(z@Wd1+bd1) -> xhat=...@Wd2+bd2`。高斯解码下重建项 = MSE。

**负 ELBO 损失** `= 重建MSE + KL`（注意 KL 是 **加号**！）。我们手写整条反向，并用数值梯度检验守住——这是 VAE 最易错的地方。

In [ ]:
def vae_loss_and_grads(x, P, eps, beta=1.0):
    '''返回 (loss, grads, parts)。loss = 重建MSE(每样本均值) + beta*KL(每样本均值)。'''
    N = x.shape[0]
    # --- 前向 ---
    h_e = np.tanh(x @ P['W1'] + P['b1'])
    mu = h_e @ P['Wmu'] + P['bmu']
    logvar = h_e @ P['Wlv'] + P['blv']
    std = np.exp(0.5 * logvar)
    z = mu + std * eps                              # 重参数化
    h_d = np.tanh(z @ P['Wd1'] + P['bd1'])
    xhat = h_d @ P['Wd2'] + P['bd2']
    recon = np.mean(np.sum((xhat - x)**2, axis=1))  # 高斯重建 = MSE
    kl = np.mean(0.5 * np.sum(mu**2 + np.exp(logvar) - 1 - logvar, axis=1))
    loss = recon + beta * kl
    # --- 反向 ---
    g = {}
    dxhat = (2.0 / N) * (xhat - x)                  # d recon / d xhat
    g['Wd2'] = h_d.T @ dxhat; g['bd2'] = dxhat.sum(0)
    dh_d = dxhat @ P['Wd2'].T
    dz_dec = dh_d * (1 - h_d**2)                    # 过 tanh
    g['Wd1'] = z.T @ dz_dec; g['bd1'] = dz_dec.sum(0)
    dz = dz_dec @ P['Wd1'].T                        # d loss / d z (来自重建)
    # z = mu + std*eps:  dmu += dz ; dstd = dz*eps ; std=exp(0.5 logvar) -> dlogvar += 0.5*std*eps*dz
    dmu = dz.copy()
    dlogvar = dz * eps * std * 0.5
    # KL 对 mu, logvar 的梯度（每样本均值 -> 1/N）:  dKL/dmu = mu ; dKL/dlogvar = 0.5*(exp(logvar)-1)
    dmu += beta * (1.0 / N) * mu
    dlogvar += beta * (1.0 / N) * 0.5 * (np.exp(logvar) - 1.0)
    g['Wmu'] = h_e.T @ dmu; g['bmu'] = dmu.sum(0)
    g['Wlv'] = h_e.T @ dlogvar; g['blv'] = dlogvar.sum(0)
    dh_e = dmu @ P['Wmu'].T + dlogvar @ P['Wlv'].T
    dpre_e = dh_e * (1 - h_e**2)
    g['W1'] = x.T @ dpre_e; g['b1'] = dpre_e.sum(0)
    return loss, g, (recon, kl)

P = init_vae()
eps_fixed = rng.standard_normal((X.shape[0], 2))
loss, grads, (recon, kl) = vae_loss_and_grads(X, P, eps_fixed)
print('loss=%.4f (recon=%.4f, kl=%.4f)' % (loss, recon, kl))
# 数值梯度检验（固定 eps 使损失确定）：抽查几个参数
def loss_only(P_):
    return vae_loss_and_grads(X, P_, eps_fixed)[0]
maxerr = 0.0
for key in ['Wmu', 'Wlv', 'Wd2', 'W1']:
    W = P[key]; gnum = np.zeros_like(W)
    it = np.nditer(W, flags=['multi_index'])
    while not it.finished:
        i = it.multi_index; old = W[i]
        W[i] = old + 1e-6; fp = loss_only(P)
        W[i] = old - 1e-6; fm = loss_only(P)
        W[i] = old; gnum[i] = (fp - fm) / 2e-6
        it.iternext()
    err = np.max(np.abs(gnum - grads[key])); maxerr = max(maxerr, err)
    print('  grad check %-4s max|err|=%.2e' % (key, err))
assert maxerr < 1e-5, 'ELBO 反向必须与数值梯度一致'
print('✅ 完整 ELBO 前向+反向写对了（含 KL 加号、重参数梯度），可放心训练')

## 5 · 训练 VAE 并从先验采样生成

训练 = 每步重采 `ε`、算负 ELBO、手写梯度、更新。训练后**从先验 `z~N(0,I)` 采样、解码**，看生成分布是否覆盖数据的双月牙——这是 AE 做不到的！

In [ ]:
def decode(z, P):
    h_d = np.tanh(z @ P['Wd1'] + P['bd1'])
    return h_d @ P['Wd2'] + P['bd2']

def train_vae(X, beta=1.0, lr=0.02, epochs=4000, seed=0):
    P = init_vae(seed=seed)
    g = np.random.default_rng(123)
    hist = []
    for ep in range(epochs):
        eps = g.standard_normal((X.shape[0], 2))
        loss, grads, parts = vae_loss_and_grads(X, P, eps, beta=beta)
        for k in P: P[k] -= lr * grads[k]
        hist.append((loss, parts[0], parts[1]))
    return P, hist

P_t, hist = train_vae(X, beta=1.0)
L0, L1 = hist[0][0], hist[-1][0]
print('负ELBO: %.3f -> %.3f  (recon %.3f, kl %.3f)' % (L0, L1, hist[-1][1], hist[-1][2]))
assert L1 < L0 * 0.6, '训练应显著降低负 ELBO'
# 从先验采样生成
z_prior = np.random.default_rng(7).standard_normal((600, 2))
gen = decode(z_prior, P_t)
# 生成分布应覆盖数据范围（均值、各维标准差接近）
print('数据   均值', np.round(X.mean(0),2), ' std', np.round(X.std(0),2))
print('生成   均值', np.round(gen.mean(0),2), ' std', np.round(gen.std(0),2))
assert np.all(np.abs(gen.mean(0) - X.mean(0)) < 0.4), '生成均值应接近数据'
assert np.all(gen.std(0) > 0.4 * X.std(0)), '生成应有合理spread(未坍缩到一点)'
print('✅ VAE 从先验 N(0,I) 采样 -> 解码出覆盖数据分布的新样本（AE 做不到！）')

## 6 · 后验坍塌：KL 权重过大把隐变量逼废

把 β 调很大，KL 项主导，编码器被逼让 `q(z|x)≈N(0,I)`——隐编码不再携带 `x` 的信息（**后验坍塌**）。

诊断信号：**KL 项趋零**，且不同 `x` 的编码均值 `μ` 趋同（区分不开样本）。

In [ ]:
def encoder_mu(x, P):
    h = np.tanh(x @ P['W1'] + P['b1'])
    return h @ P['Wmu'] + P['bmu']

# 正常 β=1 vs 过大 β=50
P_ok, hist_ok = train_vae(X, beta=1.0, epochs=4000)
P_collapse, hist_cl = train_vae(X, beta=50.0, epochs=4000)
kl_ok = hist_ok[-1][2]
kl_cl = hist_cl[-1][2]
# 编码均值的样本间方差：坍塌时趋零（所有 x 编码到同一点）
spread_ok = encoder_mu(X, P_ok).var(0).mean()
spread_cl = encoder_mu(X, P_collapse).var(0).mean()
print('β=1   : KL=%.4f, 编码均值样本间方差=%.4f' % (kl_ok, spread_ok))
print('β=50  : KL=%.4f, 编码均值样本间方差=%.4f' % (kl_cl, spread_cl))
assert kl_cl < kl_ok, 'β 过大 -> KL 被压得更低'
assert spread_cl < spread_ok * 0.5, '坍塌时编码趋同(区分不开样本)'
print('✅ 后验坍塌复现：β 过大 -> KL≈0、编码趋同、隐变量名存实亡')

---
## ✏️ 练习 1：ELBO 的两项（重建 + KL）

实现 `neg_elbo(x, xhat, mu, logvar, beta=1.0)`：返回 `(总损失, 重建项, KL项)`。
重建用高斯/MSE（每样本平方和、再对样本平均）；KL 用闭式（每样本求和、再对样本平均）；总损失 = 重建 + β·KL。

In [ ]:
def neg_elbo(x, xhat, mu, logvar, beta=1.0):
    # TODO: recon = mean_n sum_D (xhat-x)^2
    #       kl   = mean_n [ 0.5 * sum_d (mu^2 + exp(logvar) - 1 - logvar) ]
    #       return recon + beta*kl, recon, kl
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
xx = rng.standard_normal((10, 2))
# 完美重建 + 后验=先验 -> 损失为 0
tot, rc, kl_ = neg_elbo(xx, xx, np.zeros((10,2)), np.zeros((10,2)))
assert abs(tot) < 1e-12 and abs(rc) < 1e-12 and abs(kl_) < 1e-12
# β 放大只影响 KL 那部分
mu_, lv_ = rng.standard_normal((10,2)), rng.standard_normal((10,2))
t1, _, k1 = neg_elbo(xx, xx+0.1, mu_, lv_, beta=1.0)
t2, _, k2 = neg_elbo(xx, xx+0.1, mu_, lv_, beta=3.0)
assert abs(k1 - k2) < 1e-9, 'KL 项本身与 β 无关'
assert abs((t2 - t1) - 2*k1) < 1e-9, 'β 从1->3 使总损失增加 2*KL'
print('✅ 练习 1 通过：ELBO 两项 + β 加权正确')

## ✏️ 练习 2：重参数化

实现 `reparam(mu, logvar, eps)` 返回 `z=μ+σ⊙ε`，并实现它对 `mu`/`logvar` 的梯度`reparam_grads(dz, logvar, eps)` 返回 `(dmu, dlogvar)`（给定上游 `dz`）。
提示：`dmu=dz`，`σ=exp(0.5logvar)`，`z=μ+σε` -> `dlogvar = dz*ε*σ*0.5`。

In [ ]:
def reparam(mu, logvar, eps):
    # TODO: z = mu + exp(0.5*logvar)*eps
    raise NotImplementedError

def reparam_grads(dz, logvar, eps):
    # TODO: dmu = dz ; dlogvar = dz * eps * exp(0.5*logvar) * 0.5
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
mu_, lv_ = np.array([0.5,-1.0]), np.log(np.array([0.4,2.0]))
e_ = np.array([0.3, -0.7])
z_ = reparam(mu_, lv_, e_)
assert np.allclose(z_, mu_ + np.exp(0.5*lv_)*e_)
# 梯度对拍数值（上游 dz=1）
dz_ = np.ones(2)
dmu_, dlv_ = reparam_grads(dz_, lv_, e_)
assert np.allclose(dmu_, np.ones(2))
num_dlv = (reparam(mu_, lv_+1e-6, e_) - reparam(mu_, lv_-1e-6, e_)) / 2e-6
assert np.allclose(dlv_, num_dlv, atol=1e-6), 'dlogvar 应与数值一致'
print('✅ 练习 2 通过：重参数化前向 + 梯度正确')

## ✏️ 练习 3：闭式 KL 与其梯度

实现 `kl_term(mu, logvar)`（每样本求和、再对样本平均）与其梯度 `kl_grads(mu, logvar, N)`，返回 `(dmu, dlogvar)`。提示：对每样本均值，`dKL/dmu = mu/N`，`dKL/dlogvar = 0.5*(exp(logvar)-1)/N`。

In [ ]:
def kl_term(mu, logvar):
    # TODO: mean_n [ 0.5*sum_d (mu^2 + exp(logvar) - 1 - logvar) ]
    raise NotImplementedError

def kl_grads(mu, logvar):
    # TODO: N=mu.shape[0]; dmu = mu/N ; dlogvar = 0.5*(exp(logvar)-1)/N
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
mu_ = rng.standard_normal((8, 2)); lv_ = 0.3*rng.standard_normal((8, 2))
k = kl_term(mu_, lv_)
assert k >= 0, 'KL >= 0'
dmu_, dlv_ = kl_grads(mu_, lv_)
# 数值检验
def klf(m, l): return kl_term(m, l)
gmu = np.zeros_like(mu_)
for i in np.ndindex(mu_.shape):
    o = mu_[i]; mu_[i]=o+1e-6; fp=klf(mu_,lv_); mu_[i]=o-1e-6; fm=klf(mu_,lv_); mu_[i]=o
    gmu[i]=(fp-fm)/2e-6
assert np.allclose(gmu, dmu_, atol=1e-7), 'dmu 应与数值一致'
print('✅ 练习 3 通过：闭式 KL 及其梯度正确')

## ✏️ 练习 4：从先验采样生成

实现 `sample_vae(P, n, decode_fn, seed=0)`：从先验 `z~N(0,I)`（2 维）采 `n` 个，用 `decode_fn(z, P)` 解码，返回生成样本。这是 VAE「生成」的本体。

In [ ]:
def sample_vae(P, n, decode_fn, seed=0):
    # TODO: z = N(0,1) 抽 (n,2)；返回 decode_fn(z, P)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
gen = sample_vae(P_t, 500, decode, seed=42)
assert gen.shape == (500, 2)
# 生成分布应接近数据（用直方图/矩粗略判定）
assert np.all(np.abs(gen.mean(0) - X.mean(0)) < 0.5)
assert np.all(gen.std(0) > 0.4 * X.std(0))
# 换种子应得到不同样本（确实在随机采样）
gen2 = sample_vae(P_t, 500, decode, seed=43)
assert not np.allclose(gen, gen2)
print('✅ 练习 4 通过：从先验采样 -> 解码出新样本，且随机性正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def neg_elbo(x, xhat, mu, logvar, beta=1.0):
    recon = np.mean(np.sum((xhat - x)**2, axis=1))
    kl = np.mean(0.5 * np.sum(mu**2 + np.exp(logvar) - 1 - logvar, axis=1))
    return recon + beta * kl, recon, kl

In [ ]:
# 练习 2 参考答案
def reparam(mu, logvar, eps):
    return mu + np.exp(0.5 * logvar) * eps

def reparam_grads(dz, logvar, eps):
    dmu = dz
    dlogvar = dz * eps * np.exp(0.5 * logvar) * 0.5
    return dmu, dlogvar

In [ ]:
# 练习 3 参考答案
def kl_term(mu, logvar):
    return np.mean(0.5 * np.sum(mu**2 + np.exp(logvar) - 1 - logvar, axis=1))

def kl_grads(mu, logvar):
    N = mu.shape[0]
    dmu = mu / N
    dlogvar = 0.5 * (np.exp(logvar) - 1) / N
    return dmu, dlogvar

In [ ]:
# 练习 4 参考答案
def sample_vae(P, n, decode_fn, seed=0):
    g = np.random.default_rng(seed)
    z = g.standard_normal((n, 2))
    return decode_fn(z, P)

---
## 🧪 真实配置胶囊：β 的 KL/重建权衡（对标真实 VAE 超参）

真实 VAE/β-VAE 论文里 β 是核心超参。这里在玩具数据上**扫一遍 β**，画出「β ↑ -> KL ↓、重建 ↑」的权衡曲线，对应 Higgins 等 2017 报告的现象。**无需联网，纯本地训练**。

In [ ]:
# 扫 β：记录每个 β 训练后的 (重建, KL)
betas = [0.1, 0.5, 1.0, 4.0, 16.0]
rows = []
for b in betas:
    Pb, hb = train_vae(X, beta=b, epochs=3000)
    rows.append((b, hb[-1][1], hb[-1][2]))   # (β, recon, kl)
print(f"{'β':>6} {'重建':>8} {'KL':>8}")
for b, rc, kl in rows:
    print(f'{b:>6.1f} {rc:>8.3f} {kl:>8.3f}')
recons = [r[1] for r in rows]; kls = [r[2] for r in rows]
print('✅ 真实配置就绪（β 扫描完成）')

**🧪 胶囊练习**：实现 `is_tradeoff_monotone(betas, recons, kls)`：验证随 β 增大，**KL 单调不增**（正则更强 -> 后验更贴先验 -> KL 更小）且**重建单调不减**（正则更强 -> 重建更差）。返回 bool。

In [ ]:
def is_tradeoff_monotone(betas, recons, kls):
    # TODO: 按 β 升序，检查 kls 非增 且 recons 非减（容忍 1e-2 抖动）；返回 bool
    raise NotImplementedError

In [ ]:
# 自测
ok = is_tradeoff_monotone(betas, recons, kls)
assert ok, 'β↑ 应使 KL↓ 且 重建↑（rate-distortion 权衡）'
# 极端对比：最小 β 的 KL 应明显大于最大 β 的 KL
assert kls[0] > kls[-1], '小β的KL远大于大β'
print('✅ 胶囊练习通过：复现 β-VAE 的 KL/重建权衡（rate-distortion）')

In [ ]:
# 📖 胶囊参考答案
def is_tradeoff_monotone(betas, recons, kls):
    order = np.argsort(betas)
    k = np.array(kls)[order]; r = np.array(recons)[order]
    kl_ok = all(k[i+1] <= k[i] + 1e-2 for i in range(len(k)-1))
    rc_ok = all(r[i+1] >= r[i] - 1e-2 for i in range(len(r)-1))
    return bool(kl_ok and rc_ok)

### 小结
- VAE = 概率编码器 `q(z|x)=N(μ,σ²)` + 先验 `N(0,I)` + 解码器，目标是最大化 **ELBO = 重建 − KL**。
- **重参数化** `z=μ+σε` 让梯度穿过采样；**高斯 KL 有闭式**，零方差又便宜。
- 训练后**从先验采样、解码**就能生成新样本——这是普通 AE 做不到的关键超能力。
- **后验坍塌**：KL≈0、隐变量名存实亡；**β-VAE** 用 β 调 KL/重建权衡（rate-distortion）。
- 注意 KL 在损失里是**加号**（最小化负 ELBO），符号搞反会训坏。

下一站：**模块 03 · GAN** —— 彻底放弃似然与 ELBO，用对抗的判别器定义「像不像真的」，换取锐利样本。